# Notebook 11 ordered CPU input preparation

Purpose: reduce GPU idle time by preparing independent samples concurrently. Inputs and outputs match the original training `make_batch` and evaluation `_batch` provider functions exactly. Six ordered CPU workers are the default; the original sampler remains serial. Every task uses the original per-example augmentation and mask logic and local ID/occurrence RNG. No images or labels are added, no batches are prefetched, and every future is joined before return. There is no additional replay state to checkpoint.

Tagged definition imports do not install hooks. The consuming Notebook 11 workflow must call `install_input_pipeline(workers=6)` before calibration, checkpoint-resume smoke tests, training and evaluation. Repeated same-count installation is idempotent; changing count in an existing process fails explicitly. Worker count belongs in registered configuration. Expected import runtime: less than one minute. No study starts here; checkpoints and exact resume belong to the consuming workflow under `checkpoints/<run_id>/`. Restart this notebook from the first cell. Validation: `tests/11_Encoder_Input_Validation.ipynb`.


In [ ]:
from pathlib import Path
import json, os
ROOT=Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_bootstrap={}
for _cell in json.loads((ROOT/"fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if _cell["cell_type"]=="code" and "ahr-export" in _cell.get("metadata",{}).get("tags",[]):
        exec("".join(_cell["source"]), _bootstrap)
_bootstrap["install_notebook_package"](ROOT)


In [ ]:
"""Ordered synchronous CPU preparation; original per-example formulas are reused."""
from concurrent.futures import ThreadPoolExecutor
import torch
import fmm_jepa.food101_data as DATA
import fmm_jepa.food101_evaluation as EVALUATION

BACKEND = "ordered-original-example-cpu-v1"


def _assemble_original_examples(examples):
    first=examples[0]
    context=DATA._pad_rows([item['masks']['context_indices'][0][item['masks']['context_valid'][0]] for item in examples])
    target_count=len(first['masks']['target_indices'])
    targets=[DATA._pad_rows([item['masks']['target_indices'][k][0][item['masks']['target_valid'][k][0]] for item in examples]) for k in range(target_count)]
    records=[item['mask_metadata'][0] for item in examples]
    masks={'context_indices':context[0],'context_valid':context[1],
           'target_indices':[target[0] for target in targets],'target_valid':[target[1] for target in targets],
           'metadata':{'examples':records,'grid':list(first['masks']['metadata']['grid']),'target_overlap_allowed':True}}
    return {'images':torch.cat([item['images'] for item in examples],0),
            'labels':torch.cat([item['labels'] for item in examples],0),
            'sample_ids':[item['sample_ids'][0] for item in examples],
            'occurrences':[item['occurrences'][0] for item in examples],
            'augmentation_ids':[item['augmentation_ids'][0] for item in examples],
            'mask_ids':[item['mask_ids'][0] for item in examples],
            'augmentations':[item['augmentations'][0] for item in examples],
            'masks':masks,'mask_metadata':records,'preprocessing':first['preprocessing']}


def install_input_pipeline(workers=6):
    """Patch only provider functions; sampler and checkpoint state are unchanged.

    Every task calls the original function on a single example using its original
    local ID/occurrence RNG. map preserves caller order; all tasks finish before
    padding/assembly and before return. There is no lookahead or prefetch state.
    Evaluation parallelizes original dataset __getitem__ reads in exact order.
    """
    if isinstance(workers,bool) or int(workers)!=workers or not 1<=int(workers)<=24:
        raise ValueError('Integer preprocessing worker count in [1,24] required')
    workers=int(workers)
    existing=getattr(DATA,'_fmm11_input_pipeline',None)
    if existing is not None:
        if existing['workers']!=workers:raise ValueError('Input pipeline is already installed with a different worker count')
        return dict(existing['statistics'])
    original=DATA.make_batch
    original_evaluation_batch=EVALUATION._batch
    def prepared(dataset,sample_ids,occurrences,config,training=True):
        if not sample_ids or len(sample_ids)!=len(occurrences):
            raise ValueError('Require nonempty paired image identities and occurrence numbers')
        if workers==1:return original(dataset,sample_ids,occurrences,config,training=training)
        pairs=list(zip(sample_ids,occurrences))
        def one(pair):return original(dataset,[pair[0]],[pair[1]],config,training=training)
        with ThreadPoolExecutor(max_workers=workers,thread_name_prefix='fmm11-input') as pool:
            examples=list(pool.map(one,pairs))
        return _assemble_original_examples(examples)
    def evaluation_batch(dataset,start,stop):
        if workers==1:return original_evaluation_batch(dataset,start,stop)
        with ThreadPoolExecutor(max_workers=workers,thread_name_prefix='fmm11-evaluation') as pool:
            items=list(pool.map(dataset.__getitem__,range(start,stop)))
        return (torch.stack([item['image'] for item in items]),
                torch.tensor([int(item['label']) for item in items],dtype=torch.long),
                [str(item['sample_id']) for item in items])
    statistics={'status':'INSTALLED','backend':BACKEND,'workers':workers,'preprocessing_workers':workers,
                'dataloader_workers':0,'prefetch':False,'sampling':'Original ConsumedSampler remains serial',
                'rng':'Original independent per-sample ID/occurrence NumPy and Torch generators',
                'training_provider':'fmm_jepa.food101_data.make_batch',
                'evaluation_provider':'fmm_jepa.food101_evaluation._batch','ordered':True,'all_futures_join_before_return':True}
    DATA.make_batch=prepared;EVALUATION._batch=evaluation_batch
    DATA._fmm11_input_pipeline={'workers':workers,'statistics':statistics,'original':original,
                                'original_evaluation_batch':original_evaluation_batch,'prepared':prepared,'evaluation_batch':evaluation_batch}
    return dict(statistics)


In [ ]:
print({"backend":BACKEND,"default_workers":6,"installed":False,"training_started":False})